In [0]:
from pyspark.sql.functions import col

df_oc = spark.table(
    "chilecompra.silver.ordenes_compra"
)

In [0]:
df_compras_validas = (
    df_oc
    .filter(
        col("fecha_envio").isNotNull()
        & col("monto_total_clp").isNotNull()
        & (col("codigo_estado") != 9)
    )
)

In [0]:
from pyspark.sql.functions import (
    year,
    month,
    countDistinct,
    sum,
    avg
)

df_proveedores = (
    df_compras_validas
    .groupBy(
        year(col("fecha_envio")).alias("anio"),
        month(col("fecha_envio")).alias("mes"),
        col("codigo_proveedor"),
        col("nombre_proveedor")
    )
    .agg(
        countDistinct("codigo").alias("cantidad_ordenes"),
        sum("monto_total_clp").alias("monto_total_ordenes_clp"),
        avg("monto_total_clp").alias("monto_promedio_orden_clp"),
        countDistinct("codigo_organismo_publico").alias("organismos_distintos")
    )
    .orderBy("anio", "mes", col("monto_total_ordenes_clp").desc())
)

In [0]:
total_rows = df_proveedores.count()

null_keys = (
    df_proveedores
    .filter(
        col("anio").isNull()
        | col("mes").isNull()
        | col("codigo_proveedor").isNull()
    )
    .count()
)

distinct_keys = (
    df_proveedores
    .select(
        "anio",
        "mes",
        "codigo_proveedor"
    )
    .distinct()
    .count()
)

negative_amounts = (
    df_proveedores
    .filter(col("monto_total_ordenes_clp") < 0)
    .count()
)

if total_rows == 0:
    raise ValueError(
        "DQ FAILED: provider aggregation produced 0 rows"
    )

if null_keys > 0:
    raise ValueError(
        f"DQ FAILED: {null_keys} rows have NULL business keys"
    )

if total_rows != distinct_keys:
    raise ValueError(
        "DQ FAILED: duplicated year/month/provider keys"
    )

if negative_amounts > 0:
    raise ValueError(
        f"DQ FAILED: {negative_amounts} rows have negative amounts"
    )

print(
    f"Pre-write provider Gold DQ passed: {total_rows} rows"
)

In [0]:
target_table = "chilecompra.gold.ordenes_por_proveedor_mensual"

(
    df_proveedores
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

In [0]:
df_gold_proveedores = spark.table(target_table)

gold_rows = df_gold_proveedores.count()

gold_distinct_keys = (
    df_gold_proveedores
    .select(
        "anio",
        "mes",
        "codigo_proveedor"
    )
    .distinct()
    .count()
)

gold_null_keys = (
    df_gold_proveedores
    .filter(
        col("anio").isNull()
        | col("mes").isNull()
        | col("codigo_proveedor").isNull()
    )
    .count()
)

if gold_rows != df_proveedores.count():
    raise ValueError(
        "DQ FAILED: Gold row count does not match source aggregation"
    )

if gold_rows != gold_distinct_keys:
    raise ValueError(
        "DQ FAILED: duplicated year/month/provider keys in Gold"
    )

if gold_null_keys > 0:
    raise ValueError(
        f"DQ FAILED: {gold_null_keys} Gold rows have NULL business keys"
    )

print(
    f"Gold ordenes_por_proveedor_mensual DQ passed: "
    f"{gold_rows} rows"
)